# 9 · Break it, watch nothing fail, and find it anyway

Everything works. That is the worst moment to stop, because you have not yet
seen **the failure this whole course is built around**.

> A field moves upstream. The pipeline does not fail. No run errors. Every row
> count stays exactly the same. And a number quietly stops being true.

You are going to cause it, on purpose, and then find it.

In [ ]:
import sys; sys.path.insert(0, '.')
from nb import show, sql, fetch, run, counts

import psycopg
from pipelines.lib.config import dsn, SCHEMA

---

## Step 1 · Write down what "healthy" looks like, before you break anything

You cannot notice a change you never measured. Take the reading first.

In [ ]:
def reading():
    with psycopg.connect(dsn()) as c:
        return dict(zip(
            ['driver_app_rows', 'with_surge', 'avg_surge', 'gold_days', 'gold_avg_surge'],
            c.execute(f"""
                SELECT (SELECT count(*)  FROM {SCHEMA}.bronze_driver_app),
                       (SELECT count(surge) FROM {SCHEMA}.bronze_driver_app),
                       (SELECT round(avg(surge), 3) FROM {SCHEMA}.bronze_driver_app),
                       (SELECT count(*) FROM {SCHEMA}.gold_daily),
                       (SELECT round(avg(avg_surge), 3) FROM {SCHEMA}.gold_daily)
            """).fetchone()))

before = reading()
for k, v in before.items():
    print(f'  {k:18} {v}')

In [ ]:
run('-m', 'signals.board')

### Five green, one red, and the red one is real

`records_held` is above zero because the payment processor genuinely sends
settlements with an empty status, and the contract genuinely refuses them. That
is an open issue somebody owns, not a bug in the board.

**Write the number down.** The point of the next few cells is not that a light
goes red, it is that **a second, different light** goes red, for a reason
nothing else in the estate would have told you.

---

## Step 2 · Now break it

`break_it.py` simulates one thing: the mobile team ships a release that moves
the surge field, so our reader finds nothing at any path it knows.

It does **not** delete rows. It does **not** corrupt anything. It does exactly
what a real rename does: the value stops arriving.

In [ ]:
run('break_it.py')

## Look at what just happened. Or rather, at what did not.

In [ ]:
after = reading()

print(f'{"":18} {"before":>12} {"after":>12}')
for k in before:
    b, a = before[k], after[k]
    flag = '' if b == a else '   <- moved'
    print(f'  {k:18} {str(b):>12} {str(a):>12}{flag}')

### Read the first row again

**`driver_app_rows` did not change.**

Every row count check you could write is still green. Nothing failed. No
pipeline errored. There is no log line anywhere describing this.

The only thing that changed is that a column stopped being filled in.

In [ ]:
run('-m', 'pipelines.p7_silver_rides')

In [ ]:
run('-m', 'pipelines.p8_gold_daily')

Both pipelines say **ok**. Both wrote the same number of rows they always write.

---

## Step 3 · Now ask the board

In [ ]:
run('-m', 'signals.board')

### There it is

**Two breaches now, not one.** `surge_missing_pct` moved from 0 to fifteen
percent, and the line under the board says **whose it is**: pricing.

That is the entire value of the exercise. Not that a check went red, but that:

| | |
|---|---|
| **what moved** | one named number, with its normal value beside it |
| **whose it is** | pricing, not "the data team" |
| **what to say** | *"the share of driver app records with no surge value went from 0% to 30% on Tuesday"* |

Compare that to the alternative, which is somebody in finance noticing a
quarterly number looks odd, six weeks later.

## And what did it cost downstream?

In [ ]:
sql(f"""
    SELECT trip_date, rides, avg_surge, revenue
    FROM {SCHEMA}.gold_daily ORDER BY trip_date DESC
""", 'gold_daily, after the break')

### Notice which number moved and which did not

**`rides` is unchanged.** **`revenue` is unchanged.** Only `avg_surge` moved,
and it moved because it is now averaging **fewer** rows, not wrong ones.

That is `avg()` ignoring nulls, which you met in notebook 8. A missing field
does not drag an average down. It shrinks the population behind it silently.

**Which is worse**, because a number that moved is visible and a number computed
from half the data is not.

---

## Step 4 · Put it back

In [ ]:
run('break_it.py', '--fix')

In [ ]:
run('-m', 'pipelines.p7_silver_rides')
run('-m', 'pipelines.p8_gold_daily')

In [ ]:
run('-m', 'signals.board')

**Back to one breach: the one that was already there when you started.**

`surge_missing_pct` is green again and the numbers match the reading you took at
the start. That is what "fixed" looks like: not an empty board, but a board that
says exactly what it said before, and nothing more.

---

## Step 5 · The reset that always works

Every demo in this course is repeatable, because there is one command that puts
the warehouse back to nothing.

In [ ]:
sql(f"""
    SELECT reason, count(*) AS records
    FROM {SCHEMA}.quarantine GROUP BY 1 ORDER BY 2 DESC LIMIT 5
""", 'what is being held right now, and why')

### What reset actually does, and what it deliberately does not

| | |
|---|---|
| drops the `teach` schema | every table you built, gone |
| deletes the Kafka consumer groups | **the bit people forget** |
| touches the source systems | **never.** `kerb.trips`, Mongo, MinIO and the partner API are untouched |

The Kafka part is worth saying out loud. **Our position in the stream lives on
the broker, not in our database.** Drop the tables without deleting the consumer
group and the pipeline wakes up believing it has already read everything, reads
zero messages, reports success, and leaves you with an empty table and no error.

That bug cost an hour while this course was being built.

```bash
python cli.py reset          # then: python cli.py run all
```

## Prove it is repeatable

Run the reset, then rebuild everything from cold. It takes a few seconds.

In [ ]:
run('reset.py')

In [ ]:
counts()

In [ ]:
run('cli.py', 'run', 'all')

In [ ]:
counts()

In [ ]:
run('-m', 'signals.board')

**From an empty schema to a working board in one command**, back to the same
single honest breach it started with.

That is the property that makes this safe to teach with. Nothing you do in this
notebook can put the estate in a state you cannot get out of.

---

## What you learned

- **Take the reading before you break anything.** You cannot notice a change you
  never measured
- The failure that matters most is the one where **nothing fails**
- A row count check stays green through a field rename. Every time
- `avg()` ignoring nulls means a missing field **shrinks the population**, it
  does not move the average. That is harder to see, not easier
- A signal is only useful if it names **an owner** and **a normal value**
- Your position in a stream lives **on the broker**. A reset that forgets that
  is not a reset
- **Always have a way back to zero.** A demo you cannot repeat is a demo you
  will not run